# B.O.S.S. — YOLOv8 Training su dati BOSS

**Pipeline:**
1. Setup ambiente e librerie
2. Conversione Cityscapes → formato YOLO e integrazione COCO
3. Definizione classi e creazione data.yaml
4. Statistiche dataset pre-training
5. Training YOLOv8n
6. Esportazione modello (ONNX)

---
**Nota:** La valutazione quantitativa è gestita separatamente in `boss_yolo_test.ipynb`.

/kaggle/input/models/lorenzoverdura/boss-yolo-checkpoint/pytorch/default/1/last.pt

In [ ]:
import os, shutil
from pathlib import Path

IS_COLAB  = "google.colab" in str(get_ipython()) if "get_ipython" in dir() else False
IS_KAGGLE = os.environ.get("KAGGLE_KERNEL_RUN_TYPE") is not None

if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK_DIR = Path("/content/drive/MyDrive/Lavoro/code")
    os.chdir(WORK_DIR)
    print("Colab. Working dir:", os.getcwd())
elif IS_KAGGLE:
    WORK_DIR = Path("/kaggle/working")
    os.chdir(WORK_DIR)
    print("Kaggle. Working dir:", os.getcwd())
    print("Input disponibili:", list(Path("/kaggle/input").iterdir()))
else:
    print("Locale. Cella saltata.")


## Cella 1 — Installazione dipendenze
Eseguire solo la prima volta o in un nuovo ambiente.

In [ ]:
# Su Kaggle torch/torchvision/numpy/opencv sono preinstallati.
# Installa solo le librerie mancanti.
%pip install ultralytics pyyaml tqdm --quiet

# Su Colab o locale (torch non preinstallato), decommenta:
# %pip install torch torchvision --quiet
# %pip install ultralytics opencv-python Pillow matplotlib seaborn numpy pandas tqdm pyyaml --quiet


## Cella 2 — Import librerie

In [ ]:
import os                         # operazioni su file e directory
import shutil                     # copia/spostamento file
import json                       # parsing file JSON (annotazioni Cityscapes)
import glob                       # ricerca file con pattern
from pathlib import Path          # gestione percorsi cross-platform

import numpy as np                # operazioni su array e calcolo numerico
import pandas as pd               # strutture dati tabellari per metriche
import cv2                        # OpenCV: lettura/scrittura immagini, disegno
from PIL import Image             # Pillow: manipolazione immagini ad alto livello

import matplotlib.pyplot as plt   # grafici 2D (curve, bar chart)
import matplotlib.patches as mpatches  # elementi grafici personalizzati nei plot
import seaborn as sns             # heatmap e grafici statistici (confusion matrix)

from tqdm import tqdm             # barre di avanzamento per loop su dataset

from ultralytics import YOLO      # modello YOLOv8: training, val, predict

import warnings
warnings.filterwarnings('ignore')

## Cella 3 — Configurazione globale
**Modifica questa cella per cambiare modello, classi o percorsi.**

In [ ]:
# ============================================================
# CONFIGURAZIONE — modifica questi valori per adattare la pipeline
# ============================================================

MODEL_PATH = "yolov8n.pt"

import torch

IS_KAGGLE = os.environ.get("KAGGLE_KERNEL_RUN_TYPE") is not None
if IS_KAGGLE:
    INPUT_DIR = Path("/kaggle/input")
    BASE_DIR  = Path("/kaggle/working")
else:
    INPUT_DIR = Path(".").resolve()
    BASE_DIR  = Path(".").resolve()

DATASET_DIR = BASE_DIR / "dataset_boss"

if IS_KAGGLE:
    CITYSCAPES_LEFTIMG = "/kaggle/input/datasets/chrisviviers/cityscapes-leftimg8bit-trainvaltest/leftImg8bit"
    CITYSCAPES_GTFINE  = "/kaggle/input/datasets/lorenzoverdura/cityscapes-gtfine-trainvaltest/gtFine"
    COCO_DIR           = "/kaggle/input/datasets/sarkisshilgevorkyan/coco-dataset-for-yolo/coco"
else:
    CITYSCAPES_LEFTIMG = BASE_DIR / "leftImg8bit"
    CITYSCAPES_GTFINE  = BASE_DIR / "gtFine"
    COCO_DIR           = BASE_DIR / "coco"

# Classi ostacoli per B.O.S.S.
BOSS_CLASSES = [
    "person",        # 0
    "car",           # 1
    "truck",         # 2
    "bus",           # 3
    "bicycle",       # 4
    "motorcycle",    # 5
    "traffic light", # 6
    "fire hydrant",  # 7
    "stop sign",     # 8
    "bench",         # 9
    "pole",          # 10
    "chair",         # 11
    "table",         # 12
    "sofa",          # 13
    "potted plant",  # 14
    "door",          # 15
    "trash bin",     # 16
    "tree",          # 17
]

NUM_CLASSES = len(BOSS_CLASSES)  # 18

EPOCHS        = 50
BATCH_SIZE    = 32
IMG_SIZE      = 640
LEARNING_RATE = 0.01
DEVICE        = "0,1" if torch.cuda.device_count() >= 2 else ("0" if torch.cuda.is_available() else "cpu")
IOU_THRESHOLD = 0.45

print(f"Modello: {MODEL_PATH}")
print(f"Classi ({NUM_CLASSES}): {BOSS_CLASSES}")
print(f"CUDA disponibile: {torch.cuda.is_available()}")
print(f"GPU disponibili: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
print(f"Device: {DEVICE}")
if IS_KAGGLE:
    print(f"Cityscapes leftImg8bit: {CITYSCAPES_LEFTIMG}")
    print(f"Cityscapes gtFine:      {CITYSCAPES_GTFINE}")
    print(f"COCO dir:               {COCO_DIR}")


## Cella 4 — Verifica percorsi dataset

**Su Kaggle:** i dataset sono già estratti in `/kaggle/input/`. La cella verifica che tutti i percorsi esistano e conta i file disponibili. Nessuna estrazione o copia viene eseguita.

**In locale/Colab:** estrae gli ZIP da BASE_DIR:
- `leftImg8bit_trainvaltest.zip` → `leftImg8bit/`
- `gtFine_trainvaltest.zip` → `gtFine/`
- `recordings.zip` → `recordings/`

`BOSS_recordings.yolov8.zip` viene gestito separatamente nella Cella 16.

In [ ]:
import zipfile

if IS_KAGGLE:
    checks = [
        ("leftImg8bit", Path(CITYSCAPES_LEFTIMG)),
        ("gtFine",      Path(CITYSCAPES_GTFINE)),
        ("GT_DIR",      Path(GT_DIR)),
        ("frames test", Path(TEST_FRAMES_DIR)),
    ]
    for name, path in checks:
        if path.exists():
            print(f"OK  {name}: {path}")
        else:
            print(f"MANCANTE  {name}: {path}")

    tf = Path(TEST_FRAMES_DIR)
    n = len(list(tf.glob("*.jpg"))) + len(list(tf.glob("*.png"))) if tf.exists() else 0
    print(f"\nFrame test trovati: {n}")
else:
    zip_specs = [
        ("leftImg8bit_trainvaltest.zip", Path(CITYSCAPES_LEFTIMG), Path(BASE_DIR)),
        ("gtFine_trainvaltest.zip",      Path(CITYSCAPES_GTFINE),  Path(BASE_DIR)),
        ("recordings.zip",               Path(BASE_DIR) / "recordings", Path(BASE_DIR)),
    ]
    for zip_name, expected_dir, extract_to in zip_specs:
        zip_path = Path(BASE_DIR) / zip_name
        if expected_dir.exists() and any(expected_dir.iterdir()):
            print(f"OK  {expected_dir.name}/ già presente")
            continue
        if not zip_path.exists():
            print(f"MANCANTE {zip_name}")
            continue
        print(f"Estrazione {zip_name} ...")
        with zipfile.ZipFile(zip_path, "r") as z:
            z.extractall(extract_to)
        print(f"  -> {expected_dir}")

## Cella 5 — Conversione Cityscapes → formato YOLO
Converte le annotazioni poligonali di Cityscapes in bounding box YOLO normalizzate.
Filtra solo le classi definite in `BOSS_CLASSES`.

**Esegui dopo la Cella 4 (verifica/estrazione Cityscapes).**

In [ ]:
CITYSCAPES_LEFTIMG = Path(CITYSCAPES_LEFTIMG)
CITYSCAPES_GTFINE  = Path(CITYSCAPES_GTFINE)
DATASET_DIR        = Path(DATASET_DIR)
GT_DIR             = Path(GT_DIR)
GT_EXTRACTED       = Path(GT_EXTRACTED)
TEST_FRAMES_DIR    = Path(TEST_FRAMES_DIR)
BASE_DIR           = Path(BASE_DIR)

# Mappa dalle classi Cityscapes alle classi B.O.S.S.
CITYSCAPES_TO_BOSS = {
    "person":        "person",
    "rider":         "person",
    "car":           "car",
    "truck":         "truck",
    "bus":           "bus",
    "bicycle":       "bicycle",
    "motorcycle":    "motorcycle",
    "pole":          "pole",
    "traffic light": "traffic light",
    "traffic sign":  "stop sign",
}

CLASS_TO_ID = {cls: idx for idx, cls in enumerate(BOSS_CLASSES)}


def cityscapes_polygon_to_yolo_bbox(polygon, img_w, img_h):
    xs = [pt[0] for pt in polygon]
    ys = [pt[1] for pt in polygon]
    x_min, x_max = min(xs), max(xs)
    y_min, y_max = min(ys), max(ys)
    x_center = ((x_min + x_max) / 2) / img_w
    y_center = ((y_min + y_max) / 2) / img_h
    width    = (x_max - x_min) / img_w
    height   = (y_max - y_min) / img_h
    return x_center, y_center, width, height


def convert_cityscapes_split(img_root_base, ann_root_base, split, output_dir):
    img_root  = Path(img_root_base) / split
    ann_root  = Path(ann_root_base) / split
    out_imgs  = Path(output_dir) / "images" / split
    out_lbls  = Path(output_dir) / "labels" / split
    out_imgs.mkdir(parents=True, exist_ok=True)
    out_lbls.mkdir(parents=True, exist_ok=True)

    if not ann_root.exists():
        print(f"  {split}: annotazioni non trovate in {ann_root} - saltato")
        return

    json_files = list(ann_root.rglob("*_gtFine_polygons.json"))
    if not json_files:
        print(f"  {split}: nessun file gtFine_polygons.json - saltato")
        return

    converted, skipped = 0, 0
    for json_path in tqdm(json_files, desc=f"Conversione {split}"):
        with open(json_path) as f:
            data = json.load(f)
        img_w, img_h = data["imgWidth"], data["imgHeight"]

        lines = []
        for obj in data["objects"]:
            city_label = obj["label"]
            if city_label not in CITYSCAPES_TO_BOSS:
                continue
            class_id = CLASS_TO_ID[CITYSCAPES_TO_BOSS[city_label]]
            polygon  = obj["polygon"]
            if len(polygon) < 3:
                continue
            xc, yc, w, h = cityscapes_polygon_to_yolo_bbox(polygon, img_w, img_h)
            if w < 0.005 or h < 0.005:
                continue
            lines.append(f"{class_id} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}")

        if not lines:
            skipped += 1
            continue

        city = json_path.parent.name
        stem = json_path.name.replace("_gtFine_polygons.json", "")
        img_name = f"{stem}_leftImg8bit.png"
        img_src  = img_root / city / img_name
        if not img_src.exists():
            skipped += 1
            continue

        shutil.copy(img_src, out_imgs / img_name)
        (out_lbls / img_name.replace(".png", ".txt")).write_text("\n".join(lines))
        converted += 1

    print(f"  {split}: {converted} immagini convertite, {skipped} saltate")


if CITYSCAPES_LEFTIMG.exists() and CITYSCAPES_GTFINE.exists():
    for split in ["train", "val"]:
        convert_cityscapes_split(CITYSCAPES_LEFTIMG, CITYSCAPES_GTFINE, split, DATASET_DIR)
    print("Conversione completata.")
else:
    print(f"Cityscapes non trovato. leftImg8bit: {CITYSCAPES_LEFTIMG} | gtFine: {CITYSCAPES_GTFINE}")

## Cella 6 — Integrazione dataset COCO per classi mancanti

Cityscapes copre solo alcune delle classi B.O.S.S. Le classi presenti in COCO vengono
integrate filtrando il dataset COCO già in formato YOLO e remappando i class ID.

Mappa COCO ID → BOSS:

| COCO ID | Nome COCO | BOSS classe |
|---------|-----------|-------------|
| 10 | fire hydrant | fire hydrant |
| 13 | bench | bench |
| 56 | chair | chair |
| 60 | dining table | table |
| 57 | couch | sofa |
| 58 | potted plant | potted plant |

Le classi `door`, `trash bin`, `tree` non sono in COCO — verranno aggiunte con dataset separati.

In [ ]:
# Mappa COCO class ID (80 classi) → BOSS class ID
# Solo le classi COCO presenti in BOSS_CLASSES e assenti in Cityscapes
COCO_TO_BOSS_ID = {
    10: CLASS_TO_ID["fire hydrant"],
    13: CLASS_TO_ID["bench"],
    56: CLASS_TO_ID["chair"],
    57: CLASS_TO_ID["sofa"],
    58: CLASS_TO_ID["potted plant"],
    60: CLASS_TO_ID["table"],
}

COCO_DIR = Path(COCO_DIR)


def integrate_coco_split(split):
    coco_img_dir   = COCO_DIR / "images" / f"{split}2017"
    coco_label_dir = COCO_DIR / "labels" / f"{split}2017"
    out_img_dir    = DATASET_DIR / "images" / split
    out_lbl_dir    = DATASET_DIR / "labels" / split
    out_img_dir.mkdir(parents=True, exist_ok=True)
    out_lbl_dir.mkdir(parents=True, exist_ok=True)

    if not coco_label_dir.exists():
        print(f"  {split}: labels COCO non trovate in {coco_label_dir}")
        return

    copied, skipped = 0, 0
    for lf in tqdm(list(coco_label_dir.glob("*.txt")), desc=f"COCO {split}"):
        lines = lf.read_text().strip().splitlines()
        new_lines = []
        for line in lines:
            parts = line.strip().split()
            if not parts:
                continue
            coco_id = int(parts[0])
            if coco_id not in COCO_TO_BOSS_ID:
                continue
            parts[0] = str(COCO_TO_BOSS_ID[coco_id])
            new_lines.append(" ".join(parts))

        if not new_lines:
            skipped += 1
            continue

        # Copia immagine e scrivi label remappata
        img_src = coco_img_dir / (lf.stem + ".jpg")
        if not img_src.exists():
            skipped += 1
            continue

        shutil.copy(img_src, out_img_dir / img_src.name)
        (out_lbl_dir / lf.name).write_text("\n".join(new_lines))
        copied += 1

    print(f"  {split}: {copied} immagini integrate, {skipped} saltate")


if COCO_DIR.exists():
    for split in ["train", "val"]:
        integrate_coco_split(split)
    print("Integrazione COCO completata.")
else:
    print(f"COCO non trovato in {COCO_DIR} — cella saltata.") 

## Cella 7 — Creazione file data.yaml per YOLOv8

In [ ]:
# Genera il file data.yaml richiesto da YOLOv8 per il training.
# Contiene i percorsi dei dataset e la definizione delle classi.

import yaml

data_yaml = {
    "path": str(DATASET_DIR),                              # percorso radice del dataset
    "train": "images/train",                              # cartella immagini training
    "val":   "images/val",                                # cartella immagini validazione
    "test":  "images/test",                               # cartella immagini test
    "nc":    NUM_CLASSES,                                 # numero di classi
    "names": BOSS_CLASSES,                                # lista nomi classi
}

yaml_path = DATASET_DIR / "data.yaml"
DATASET_DIR.mkdir(parents=True, exist_ok=True)

with open(yaml_path, "w") as f:
    yaml.dump(data_yaml, f, default_flow_style=False, allow_unicode=True)

print(f"data.yaml scritto in: {yaml_path}")
print(yaml.dump(data_yaml, default_flow_style=False))

## Cella 8 — Statistiche dataset pre-training

In [ ]:
# Analizza la distribuzione delle classi nel dataset prima del training.
# Importante per individuare class imbalance che potrebbe degradare le performance.

def count_class_distribution(label_dir):
    """
    Conta le occorrenze di ogni classe in tutti i file .txt YOLO di una cartella.
    Restituisce un dict {class_id: count}.
    """
    counts = {i: 0 for i in range(NUM_CLASSES)}
    label_files = list(Path(label_dir).rglob("*.txt"))

    # Per ogni file di annotazione: incrementa il contatore per ogni riga (= un oggetto)
    for lf in label_files:
        for line in lf.read_text().strip().split("\n"):
            if line.strip():               # salta righe vuote
                class_id = int(line.split()[0])   # primo valore = ID classe
                if class_id in counts:
                    counts[class_id] += 1
    return counts


train_counts = count_class_distribution(DATASET_DIR / "labels" / "train")
val_counts   = count_class_distribution(DATASET_DIR / "labels" / "val")

# Costruisce DataFrame per visualizzazione tabellare
df_dist = pd.DataFrame({
    "Classe":   BOSS_CLASSES,
    "Train":    [train_counts[i] for i in range(NUM_CLASSES)],
    "Val":      [val_counts[i]   for i in range(NUM_CLASSES)],
})
df_dist["Totale"] = df_dist["Train"] + df_dist["Val"]

print(df_dist.to_string(index=False))

# Grafico a barre della distribuzione delle classi nel training set
fig, ax = plt.subplots(figsize=(14, 5))
colors = plt.cm.tab20.colors[:NUM_CLASSES]   # colore diverso per ogni classe
bars = ax.bar(BOSS_CLASSES, df_dist["Train"], color=colors)
ax.set_title("Distribuzione classi — Training set", fontsize=14)
ax.set_xlabel("Classe")
ax.set_ylabel("Numero istanze")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(BASE_DIR / "plot_class_distribution.png", dpi=150)
plt.show()

## Cella 9 — Training YOLOv8
Il training salva automaticamente i pesi migliori in `runs/detect/boss_yolo/weights/best.pt`.

In [ ]:
#=== TRAINING COMPLETATO — cella commentata ===
# Fine-tuning del modello YOLOv8 sul dataset B.O.S.S.
# Tutti i parametri sono configurabili nella cella 3.

# Per riprendere un training interrotto: imposta RESUME = True
# YOLOv8 carica last.pt e continua dalle epoche rimanenti
RESUME = True

if IS_KAGGLE and RESUME:
    LAST_PT = Path("/kaggle/input/models/lorenzoverdura/boss-yolo-checkpoint/pytorch/default/1/last.pt")
    print(f"Checkpoint: {LAST_PT} — esiste: {LAST_PT.exists()}")
else:
    LAST_PT = BASE_DIR / "runs/detect/boss_yolo/weights/last.pt"

if RESUME and LAST_PT.exists():
    print(f"Riprendendo training da: {LAST_PT}")
    model = YOLO(str(LAST_PT))
    results_train = model.train(resume=True)
else:
    if RESUME:
        print("ATTENZIONE: last.pt non trovato — controlla di aver aggiunto il model al notebook.")
    model = YOLO(MODEL_PATH)   # carica il modello di partenza (pre-trained su COCO)
    results_train = model.train(
        data      = str(yaml_path),      # file data.yaml con percorsi e classi
        epochs    = EPOCHS,              # numero di epoche di training
        batch     = BATCH_SIZE,          # immagini per batch
        imgsz     = IMG_SIZE,            # risoluzione input (640x640)
        lr0       = LEARNING_RATE,       # learning rate iniziale
        device    = DEVICE,              # GPU/CPU
        name      = "boss_yolo",         # nome cartella di output in runs/detect/
        patience  = 15,                  # early stopping: ferma se val/mAP non migliora per 15 epoche
        save      = True,                # salva best.pt e last.pt dopo ogni epoca
        plots     = True,                # genera grafici training automaticamente
        verbose   = True,
    )

BEST_MODEL_PATH = Path(results_train.save_dir) / "weights" / "best.pt"
print(f"Modello migliore: {BEST_MODEL_PATH}")
print(f"Esiste: {BEST_MODEL_PATH.exists()}")


## Cella 9b — Recupero path modello dopo training

In [ ]:
import glob as _glob

# Cerca prima nella working dir (se il training è stato eseguito in questa sessione)
found_pts = _glob.glob('/kaggle/working/runs/detect/**/weights/*.pt', recursive=True)
for p in sorted(found_pts):
    print(p)

best_candidates = [p for p in found_pts if p.endswith('best.pt')]

if best_candidates:
    BEST_MODEL_PATH = Path(best_candidates[0])
    print(f"\nBEST_MODEL_PATH (da working): {BEST_MODEL_PATH}")
else:
    # Fallback: carica dal dataset Kaggle con i pesi salvati
    CHECKPOINT_DIR = Path("/kaggle/input/models/lorenzoverdura/boss-yolo-checkpoint/pytorch/default/1")
    best_ckpt = CHECKPOINT_DIR / "best.pt"
    last_ckpt = CHECKPOINT_DIR / "last.pt"

    if best_ckpt.exists():
        BEST_MODEL_PATH = best_ckpt
        print(f"\nBEST_MODEL_PATH (da checkpoint dataset): {BEST_MODEL_PATH}")
    elif last_ckpt.exists():
        BEST_MODEL_PATH = last_ckpt
        print(f"\nBEST_MODEL_PATH (last.pt da checkpoint dataset): {BEST_MODEL_PATH}")
    else:
        raise FileNotFoundError(
            f"Nessun .pt trovato né in working né in {CHECKPOINT_DIR}. "
            "Esegui il training oppure verifica che il dataset boss-yolo-checkpoint sia allegato."
        )


## Cella 10 — Esportazione modello (ONNX)

Esporta `best.pt` in formato **ONNX** per il deployment su dispositivo Edge (Jetson, OpenVINO, ONNX Runtime).

- `simplify=True`: ottimizza il grafo rimuovendo operatori ridondanti
- `opset=12`: compatibile con TensorRT 8+ e la maggior parte dei runtime Edge
- `dynamic=False`: dimensioni fisse per massima ottimizzazione hardware

Il file `.onnx` viene salvato nella stessa cartella di `best.pt`.

In [ ]:
# # ============================================================
# # Cella 10 — Esportazione modello in formato ONNX
# # ============================================================
# export_model = YOLO(str(BEST_MODEL_PATH))

# onnx_path = export_model.export(
#     format   = "onnx",
#     imgsz    = IMG_SIZE,
#     dynamic  = False,   # dimensioni input fisse per ottimizzazione hardware
#     simplify = True,    # semplifica il grafo ONNX (rimuove operatori ridondanti)
#     opset    = 12,      # compatibile con TensorRT 8+ e ONNX Runtime
# )

# print(f"Modello esportato: {onnx_path}")